# Consequences of Duplicate Labels

In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Methods like reindex(), don't work with duplicates present
# The following raises:
# ValueError: cannot reindex on an axis with duplicate labels
s1 = pd.Series([0, 1, 2], index=["a", "b", "b"])
s1.reindex(["a", "b", "c"])

ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
# A DataFrame with duplicates in the columns:
df1 = pd.DataFrame([[0, 1, 2], [3, 4, 5]], columns=["A", "A", "B"])
df1

,A,A,B
0,0,1,2
1,3,4,5


In [ ]:
# Slicing column B, returns a Series:
df1["B"]

0    2
1    5
Name: B, dtype: int64

In [5]:
# Slicing column A (the duplicate), returns a DataFrame
df1["A"]

,A,A
0,0,1
1,3,4


In [6]:
# Same with row labels as well:
df2 = pd.DataFrame({"A": [0, 1, 2]}, index=["a", "a", "b"])
df2

,A
a,0
a,1
b,2


In [ ]:
# This returns a scalar
df2.loc["b", "A"]

np.int64(2)

In [8]:
# But this returns a Series
df2.loc["a", "A"]

a    0
a    1
Name: A, dtype: int64

# Duplicate Label Detection

In [10]:
# Check whether an Index is unique:
df2.index.is_unique

False

In [11]:
df2.columns.is_unique

True

In [12]:
# Use Index.duplicated() to return a boolean ndarray:
df2.index.duplicated()

array([False,  True, False])

In [15]:
# Use the above boolean mask to drop duplicate rows:
df2.loc[~df2.index.duplicated(), :]

,A
a,0
b,2


# Disallowing Duplicate Labels

In [ ]:
# By default duplicates are allowed. 
pd.Series([0, 1, 2], index=["a", "b", "b"])

a    0
b    1
b    2
dtype: int64

In [17]:
# To disallow them:
# .set_flags(allows_duplicate_labels=False)
pd.Series([0, 1, 2], index=["a", "b", "b"]).set_flags(allows_duplicate_labels=False)

DuplicateLabelError: Index has duplicates.
      positions
label          
b        [1, 2]

In [18]:
# Also for columns:
pd.DataFrame(
    [[0, 1, 2], [3, 4, 5]],
    columns=["A", "B", "C"],
).set_flags(allows_duplicate_labels=False)

,A,B,C
0,0,1,2
1,3,4,5


In [19]:
df = pd.DataFrame({"A": [0, 1, 2, 3]}, index=["x", "y", "X", "Y"]).set_flags(allows_duplicate_labels=False)
df

,A
x,0
y,1
X,2
Y,3


In [21]:
df.flags.allows_duplicate_labels

False

In [22]:
df2 = df.set_flags(allows_duplicate_labels=True)
df2.flags.allows_duplicate_labels

True

In [23]:
df2.flags.allows_duplicate_labels = False
df2.flags.allows_duplicate_labels

False

In [24]:
# Rename columns or index labels
df.rename(str.upper)

DuplicateLabelError: Index has duplicates.
      positions
label          
X        [0, 2]
Y        [1, 3]

# Duplicate Label Propagation

In [25]:
# Disallowing duplicates is preserved through operations
s1 = pd.Series(0, index=["a", "b"]).set_flags(allows_duplicate_labels=False)
s1

a    0
b    0
dtype: int64

In [26]:
s1.head().rename({"a": "b"})

DuplicateLabelError: Index has duplicates.
      positions
label          
b        [0, 1]